In [1]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler , OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
import seaborn as sns
from sklearn.compose import ColumnTransformer
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from lazypredict.Supervised import LazyRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import RandomizedSearchCV, KFold

In [2]:
df = pd.read_csv('spotify.tsv' , sep = '\t')

In [3]:
df.head()

,Date Inserted,Reporting Date,Sale Month,Store,Artist,Title,ISRC,UPC,Quantity,Team Percentage,Song/Album,Country of Sale,Songwriter Royalties Withheld (USD),Earnings (USD),Recoup (USD)
0,2026-03-04,2026-03-04,2025-11,Deezer,jarali4.wav,nE?! 1960's,QT6EF2584101,199956793200,3,100,Song,TR,0,0.00,0.00
1,2026-03-04,2026-03-04,2025-11,Deezer,jarali4.wav,tak tak tak 1960's,QT6EE2536512,199939516277,24,100,Song,DE,0,0.31,0.00
2,2026-03-04,2026-03-04,2025-11,Deezer,jarali4.wav,tak tak tak 1960's,QT6EE2536512,199939516277,2,100,Song,FR,0,0.03,0.00
3,2026-03-04,2026-03-04,2025-11,Deezer,jarali4.wav,tak tak tak 1960's,QT6EE2536512,199939516277,1,100,Song,NL,0,0.02,0.00
4,2026-03-04,2026-03-04,2025-11,Deezer,jarali4.wav,tak tak tak 1960's,QT6EE2536512,199939516277,110,100,Song,TR,0,0.08,0.00


In [4]:
df.drop(columns = ['Date Inserted','Reporting Date', 'Sale Month', 'Artist', 'ISRC', 'UPC', 'Team Percentage', 'Song/Album', 'Songwriter Royalties Withheld (USD)', 'Recoup (USD)'], inplace = True)

In [5]:
df.head()

,Store,Title,Quantity,Country of Sale,Earnings (USD)
0,Deezer,nE?! 1960's,3,TR,0.00
1,Deezer,tak tak tak 1960's,24,DE,0.31
2,Deezer,tak tak tak 1960's,2,FR,0.03
3,Deezer,tak tak tak 1960's,1,NL,0.02
4,Deezer,tak tak tak 1960's,110,TR,0.08


In [6]:
df.isna().sum()

Store              0
Title              0
Quantity           0
Country of Sale    0
Earnings (USD)     0
dtype: int64

In [7]:
df[['Title','Store','Country of Sale']].nunique()

Title                8
Store               19
Country of Sale    124
dtype: int64

In [8]:
X = df.drop(columns = 'Earnings (USD)')
y = df['Earnings (USD)']

In [9]:
X_train , X_test , y_train , y_test = train_test_split(
    X , y , random_state = 42 , test_size = 0.2
)

In [10]:
cat_cols = X.select_dtypes(include = 'object').columns.tolist()
num_cols = X.select_dtypes(include = ['float64','int64']).columns.tolist()


In [11]:
cat_cols

['Store', 'Title', 'Country of Sale']

In [12]:
num_cols

['Quantity']

In [13]:
num_pip = Pipeline([
    ('scale', StandardScaler())
]
)

In [18]:
cat_pip = Pipeline([
    ('one', OneHotEncoder(handle_unknown = 'ignore', sparse_output = False))
])

In [19]:
data = ColumnTransformer([
    ('cat', cat_pip , cat_cols),
    ('num', num_pip, num_cols)
])

In [20]:
X_train_p = data.fit_transform(X_train)
X_test_p  = data.transform(X_test)

In [21]:
reg = LazyRegressor(verbose=0, ignore_warnings=True)
models, preds = reg.fit(X_train_p, X_test_p, y_train, y_test)
models.head(10)

  0%|          | 0/42 [00:00<?, ?it/s]

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002016 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 249
[LightGBM] [Info] Number of data points in the train set: 1712, number of used features: 50
[LightGBM] [Info] Start training from score 0.132909


,Adjusted R-Squared,R-Squared,RMSE,Time Taken
Model,,,,
XGBRegressor,0.93,0.96,0.16,0.20
BaggingRegressor,0.84,0.90,0.25,0.07
RandomForestRegressor,0.83,0.89,0.26,0.70
HuberRegressor,0.76,0.84,0.31,0.07
LinearSVR,0.75,0.84,0.31,0.42
SGDRegressor,0.74,0.83,0.32,0.01
Lars,0.74,0.83,0.32,0.02
LinearRegression,0.74,0.83,0.32,0.02
TransformedTargetRegressor,0.74,0.83,0.32,0.02
